# Week 1: Civic Grievance Data Preprocessing
**Author:** OM Gadekar  
**Date:** 2026-14-3  
**Task:** Text cleaning pipeline for 311 service request data

## Objective
Clean raw civic complaint text to prepare it for NLP analysis (word clouds, n-grams).

In [12]:
# Install required libraries (run once)
# !pip install pandas spacy
# !python -m spacy download en_core_web_sm

import pandas as pd
import re
import string
import spacy
from spacy.lang.en.stop_words import STOP_WORDS

# Load spaCy English model
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [13]:
# Load the dataset
df = pd.read_csv("NYC311data.csv")  # adjust filename to yours

# Basic inspection
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nFirst 3 rows:")
df.head(3)

C:\Users\gadek\AppData\Local\Temp\ipykernel_2600\1397343691.py:2: DtypeWarning: Columns (48,49) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("NYC311data.csv")  # adjust filename to yours


Shape: (300698, 53)

Columns: ['Unique Key', 'Created Date', 'Closed Date', 'Agency', 'Agency Name', 'Complaint Type', 'Descriptor', 'Location Type', 'Incident Zip', 'Incident Address', 'Street Name', 'Cross Street 1', 'Cross Street 2', 'Intersection Street 1', 'Intersection Street 2', 'Address Type', 'City', 'Landmark', 'Facility Type', 'Status', 'Due Date', 'Resolution Description', 'Resolution Action Updated Date', 'Community Board', 'Borough', 'X Coordinate (State Plane)', 'Y Coordinate (State Plane)', 'Park Facility Name', 'Park Borough', 'School Name', 'School Number', 'School Region', 'School Code', 'School Phone Number', 'School Address', 'School City', 'School State', 'School Zip', 'School Not Found', 'School or Citywide Complaint', 'Vehicle Type', 'Taxi Company Borough', 'Taxi Pick Up Location', 'Bridge Highway Name', 'Bridge Highway Direction', 'Road Ramp', 'Bridge Highway Segment', 'Garage Lot Name', 'Ferry Direction', 'Ferry Terminal Name', 'Latitude', 'Longitude', 'Locati

,Unique Key,Created Date,Closed Date,Agency,Agency Name,Complaint Type,Descriptor,Location Type,Incident Zip,Incident Address,...,Bridge Highway Name,Bridge Highway Direction,Road Ramp,Bridge Highway Segment,Garage Lot Name,Ferry Direction,Ferry Terminal Name,Latitude,Longitude,Location
0,32310363,12/31/2015 11:59:45 PM,01-01-16 0:55,NYPD,New York City Police Department,Noise - Street/Sidewalk,Loud Music/Party,Street/Sidewalk,10034.0,71 VERMILYEA AVENUE,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.865682,-73.923501,"(40.86568153633767, -73.92350095571744)"
1,32309934,12/31/2015 11:59:44 PM,01-01-16 1:26,NYPD,New York City Police Department,Blocked Driveway,No Access,Street/Sidewalk,11105.0,27-07 23 AVENUE,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.775945,-73.915094,"(40.775945312321085, -73.91509393898605)"
2,32309159,12/31/2015 11:59:29 PM,01-01-16 4:51,NYPD,New York City Police Department,Blocked Driveway,No Access,Street/Sidewalk,10458.0,2897 VALENTINE AVENUE,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.870325,-73.888525,"(40.870324522111424, -73.88852464418646)"


In [14]:
# Check for nulls and data types
print(df.info())
print("\nNull counts:\n", df.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300698 entries, 0 to 300697
Data columns (total 53 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   Unique Key                      300698 non-null  int64  
 1   Created Date                    300698 non-null  object 
 2   Closed Date                     298534 non-null  object 
 3   Agency                          300698 non-null  object 
 4   Agency Name                     300698 non-null  object 
 5   Complaint Type                  300698 non-null  object 
 6   Descriptor                      294784 non-null  object 
 7   Location Type                   300567 non-null  object 
 8   Incident Zip                    298083 non-null  float64
 9   Incident Address                256288 non-null  object 
 10  Street Name                     256288 non-null  object 
 11  Cross Street 1                  251419 non-null  object 
 12  Cross Street 2  

## Preprocessing Strategy

We will clean the `complaint_description` column using the following steps:
1. Lowercase all text
2. Remove URLs, email addresses
3. Remove special characters, punctuation, and digits
4. Tokenize and remove stopwords
5. Lemmatize tokens (reduce words to base form)
6. Reconstruct cleaned text and save

In [18]:
# ── Cell 6 ──────────────────────────────────────────────────────
from tqdm import tqdm   # ← add this import at the top of the cell

TEXT_COLUMN = "Descriptor"

def clean_text_basic(text):
    """Steps 1–5: fast regex cleaning (no spaCy yet)"""
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"\S+@\S+", "", text)
    text = re.sub(r"[^a-z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# Step 1: fast regex pass
print("Applying regex cleaning...")
df["temp_text"] = df[TEXT_COLUMN].apply(clean_text_basic)

# Step 2: batch lemmatization WITH progress bar
print("Applying lemmatization (batch mode)...")
cleaned_texts = []

for doc in tqdm(nlp.pipe(df["temp_text"], batch_size=500, disable=["parser", "ner"]),
                total=len(df), desc="Lemmatizing"):
    tokens = [
        token.lemma_
        for token in doc
        if token.text not in STOP_WORDS and not token.is_punct and len(token.text) > 2
    ]
    cleaned_texts.append(" ".join(tokens))

df["cleaned_text"] = cleaned_texts
df.drop(columns=["temp_text"], inplace=True)

print(f"Done! Cleaned {len(df)} rows.")
df[["Descriptor", "cleaned_text"]].head(5)



Applying regex cleaning...
Applying lemmatization (batch mode)...


Lemmatizing: 100%|██████████| 300698/300698 [04:11<00:00, 1194.56it/s]


Done! Cleaned 300698 rows.


,Descriptor,cleaned_text
0,Loud Music/Party,loud musicparty
1,No Access,access
2,No Access,access
3,Commercial Overnight Parking,commercial overnight parking
4,Blocked Sidewalk,block sidewalk


In [19]:
# Count empty rows after cleaning (may indicate data issues)
empty_count = df["cleaned_text"].str.strip().eq("").sum()
print(f"Empty rows after cleaning: {empty_count} / {len(df)}")

# Check average token count
df["token_count"] = df["cleaned_text"].apply(lambda x: len(x.split()))
print("\nToken count stats:")
print(df["token_count"].describe())


Empty rows after cleaning: 5914 / 300698

Token count stats:
count    300698.000000
mean          1.990502
std           0.900041
min           0.000000
25%           1.000000
50%           2.000000
75%           2.000000
max           4.000000
Name: token_count, dtype: float64


In [20]:
# Save cleaned dataset
df.to_csv("cleaned_311_data.csv", index=False)
print("Saved: cleaned_311_data.csv")
print(f"Total rows: {len(df)}")

Saved: cleaned_311_data.csv
Total rows: 300698


## Summary

| Step | Action | Result |
|------|--------|--------|
| Lowercase | str.lower() | Normalized case |
| URL/email removal | regex | Removed noise |
| Special char removal | regex [^a-z\s] | Clean alphabet only |
| Stopword removal | spaCy STOP_WORDS | ~30-50% token reduction |
| Lemmatization | spaCy token.lemma_ | Base word forms |

**Output:** `cleaned_311_data.csv` — ready for word cloud and n-gram analysis.